# Level 2: Instruction Tuning with QLoRA

Welcome to one of the most impactful topics in LLM tuning: **Instruction Tuning**.

Instruction tuning (also known as Supervised Fine-Tuning or SFT) is the process of training a pre-trained LLM on a dataset of high-quality instructions and their corresponding responses. This teaches the model to become more helpful, follow instructions better, and align with specific behaviors.

However, full fine-tuning (updating all of the model's billions of parameters) is incredibly memory-intensive. That's where **Parameter-Efficient Fine-Tuning (PEFT)** comes in.

In this notebook, we will focus on **QLoRA (Quantized Low-Rank Adaptation)**, a state-of-the-art PEFT method that allows us to fine-tune large models on a single consumer GPU.

### How QLoRA Works: The Magic Explained

QLoRA combines three key ideas:

1.  **4-bit Quantization**: The pre-trained model is loaded in 4-bit precision instead of the usual 16-bit or 32-bit. This dramatically reduces memory usage (e.g., a 7B parameter model goes from ~14GB to ~3.5GB in VRAM). This is handled by the `bitsandbytes` library.

2.  **Low-Rank Adaptation (LoRA)**: Instead of training the original weights, we freeze them. We then inject small, "trainable" low-rank matrices into the Transformer layers. We only train these tiny matrices, which represent a tiny fraction of the total parameters.

3.  **Paged Optimizers & Double Quantization**: Additional memory-saving tricks to handle memory spikes and further compress the model.

**The result:** We can achieve near full fine-tuning quality while using a fraction of the GPU memory.

### Step 1: Install Dependencies

We need the whole suite of Hugging Face tools for this.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets

### Step 2: Prepare the Dataset

For instruction tuning, your dataset needs to be in a specific format, typically with columns for `instruction`, `input`, and `output`. A common way to structure this for the model is a prompt template.

We'll use a small, popular dataset called `databricks/databricks-dolly-15k` which contains instruction-following records. We'll only use a small fraction of it for this demo.

In [ ]:
from datasets import load_dataset

# Load a small part of the dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:100]")

# Define a prompt template
def format_prompt(example):
    instruction = example['instruction']
    context = example['context']
    response = example['response']
    if context:
        return f"""### Instruction:
{instruction}

### Input:
{context}

### Response:
{response}"""
    else:
        return f"""### Instruction:
{instruction}

### Response:
{response}"""

# Apply the formatting
dataset = dataset.map(lambda x: {'text': format_prompt(x)})

# Let's see an example
print(dataset[0]['text'])

### Step 3: Configure QLoRA and Load the Model

This is the core of the process. We'll define our 4-bit quantization configuration and our LoRA configuration.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "mistralai/Mistral-7B-v0.1" # Using the base model for fine-tuning

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA config
lora_config = LoraConfig(
    r=16, # Rank of the update matrices. Lower rank means fewer parameters to train.
    lora_alpha=32, # Alpha is a scaling factor for the learned weights. (alpha/r)
    lora_dropout=0.05, # Dropout probability for LoRA layers.
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'] # Apply LoRA to attention projections.
)

# Load tokenizer and model with QLoRA configuration
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Set pad token to EOS token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # Automatically places the model on available devices (GPU)
)

# Wrap the model with PEFT
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

### Step 4: Train the Model

We'll use the `SFTTrainer` from the `trl` library, which simplifies the training process.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./qlora-results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1, # For demonstration, we train for a very short time
    logging_steps=10,
    fp16=True, # Use mixed precision training
    optim="paged_adamw_8bit", # Memory-efficient optimizer
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

# Start training
trainer.train()

### Step 5: Inference with the Fine-Tuned Model

After training, you can use the model for inference. The LoRA adapters are automatically applied.

In [ ]:
prompt = "### Instruction:\nWhat is QLoRA?\n\n### Response:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))